In [2]:
import os

os.chdir("/Users/rahulgupta/Desktop/C-RAG")

In [3]:
import pandas as pd

In [4]:
answers_df = pd.read_csv('app/assets/raw/datasets/stackoverflow/Answers.csv')
questions_df = pd.read_csv('app/assets/raw/datasets/stackoverflow/Questions.csv')
tags_df = pd.read_csv('app/assets/raw/datasets/stackoverflow/Tags.csv')

In [5]:
answers_df.head()

,Id,OwnerUserId,CreationDate,ParentId,Score,Body
0,497,50.0,2008-08-02T16:56:53Z,469,4,<p>open up a terminal (Applications-&gt;Utilit...
1,518,153.0,2008-08-02T17:42:28Z,469,2,<p>I haven't been able to find anything that d...
2,536,161.0,2008-08-02T18:49:07Z,502,9,<p>You can use ImageMagick's convert utility f...
3,538,156.0,2008-08-02T18:56:56Z,535,23,<p>One possibility is Hudson. It's written in...
4,541,157.0,2008-08-02T19:06:40Z,535,20,"<p>We run <a href=""http://buildbot.net/trac"">B..."


In [6]:
questions_df.head()

,Id,OwnerUserId,CreationDate,Score,Title,Body
0,469,147.0,2008-08-02T15:11:16Z,21,How can I find the full path to a font from it...,<p>I am using the Photoshop's javascript API t...
1,502,147.0,2008-08-02T17:01:58Z,27,Get a preview JPEG of a PDF on Windows?,<p>I have a cross-platform (Python) applicatio...
2,535,154.0,2008-08-02T18:43:54Z,40,Continuous Integration System for a Python Cod...,<p>I'm starting work on a hobby project with a...
3,594,116.0,2008-08-03T01:15:08Z,25,cx_Oracle: How do I iterate over a result set?,<p>There are several ways to iterate over a re...
4,683,199.0,2008-08-03T13:19:16Z,28,Using 'in' to match an attribute of Python obj...,<p>I don't remember whether I was dreaming or ...


In [7]:
tags_df.head()

,Id,Tag
0,469,python
1,469,osx
2,469,fonts
3,469,photoshop
4,502,python


In [7]:
print("Questions Shape:", questions_df.shape)
print("Answers Shape:", answers_df.shape)


Questions Shape: (607282, 6)
Answers Shape: (987122, 6)


In [8]:
print(questions_df.columns)
print(answers_df.columns)

    



Index(['Id', 'OwnerUserId', 'CreationDate', 'Score', 'Title', 'Body'], dtype='str')
Index(['Id', 'OwnerUserId', 'CreationDate', 'ParentId', 'Score', 'Body'], dtype='str')


In [8]:
from app.ingestion.preprocessing import (
    load_data,
    select_best_answers,
    merge_questions_answers
)

questions, answers = load_data(
    "app/assets/raw/datasets/stackoverflow/Questions.csv",
    "app/assets/raw/datasets/stackoverflow/Answers.csv"
)

best_answers = select_best_answers(answers)

merged_df = merge_questions_answers(
    questions,
    best_answers
)

print(merged_df.shape)
merged_df.head()

(539238, 12)


,Id_question,OwnerUserId_question,CreationDate_question,Score_question,Title,Body_question,ParentId,Id_answer,OwnerUserId_answer,CreationDate_answer,Score_answer,Body_answer
0,469,147.0,2008-08-02T15:11:16Z,21,How can I find the full path to a font from it...,<p>I am using the Photoshop's javascript API t...,469,3040,457.0,2008-08-06T03:01:23Z,12,<p>Unfortunately the only API that isn't depre...
1,502,147.0,2008-08-02T17:01:58Z,27,Get a preview JPEG of a PDF on Windows?,<p>I have a cross-platform (Python) applicatio...,502,7090,13.0,2008-08-10T08:08:33Z,25,<p>ImageMagick delegates the PDF->bitmap conve...
2,535,154.0,2008-08-02T18:43:54Z,40,Continuous Integration System for a Python Cod...,<p>I'm starting work on a hobby project with a...,535,538,156.0,2008-08-02T18:56:56Z,23,<p>One possibility is Hudson. It's written in...
3,594,116.0,2008-08-03T01:15:08Z,25,cx_Oracle: How do I iterate over a result set?,<p>There are several ways to iterate over a re...,594,595,116.0,2008-08-03T01:17:36Z,25,<p>The canonical way is to use the built-in cu...
4,683,199.0,2008-08-03T13:19:16Z,28,Using 'in' to match an attribute of Python obj...,<p>I don't remember whether I was dreaming or ...,683,57833,4702.0,2008-09-11T22:42:14Z,29,<p>Using a list comprehension would build a te...


In [10]:
print(f"-----questions Body------\n{questions_df['Body'].iloc[0]}", end="\n\n")

print(f"-----answers Body-------\n{answers_df['Body'].iloc[0]}", end="\n\n")

-----questions Body------
<p>I am using the Photoshop's javascript API to find the fonts in a given PSD.</p>

<p>Given a font name returned by the API, I want to find the actual physical font file that that font name corresponds to on the disc.</p>

<p>This is all happening in a python program running on OSX so I guess I'm looking for one of:</p>

<ul>
<li>Some Photoshop javascript</li>
<li>A Python function</li>
<li>An OSX API that I can call from python</li>
</ul>


-----answers Body-------
<p>open up a terminal (Applications-&gt;Utilities-&gt;Terminal) and type this in:</p>

<pre><code>locate InsertFontHere<br></code></pre>

<p>This will spit out every file that has the name you want.</p>

<p>Warning: there may be alot to wade through.</p>



In [11]:
from app.ingestion.html_cleaning import clean_html
sample = merged_df["Body_question"].iloc[0]


print(clean_html(sample))

I am using the Photoshop's javascript API to find the fonts in a given PSD.
Given a font name returned by the API, I want to find the actual physical font file that that font name corresponds to on the disc.
This is all happening in a python program running on OSX so I guess I'm looking for one of:
Some Photoshop javascript
A Python function
An OSX API that I can call from python


In [9]:
from app.ingestion.creat_doc import create_documents

documents = create_documents(merged_df)
print(f"Number of documents created: {len(documents)}")
print(f"Sample Document:\n{documents[0]}")

Number of documents created: 539238
Sample Document:
page_content='
TITLE:
How can I find the full path to a font from its display name on a Mac?

QUESTION:
I am using the Photoshop's javascript API to find the fonts in a given PSD.
Given a font name returned by the API, I want to find the actual physical font file that that font name corresponds to on the disc.
This is all happening in a python program running on OSX so I guess I'm looking for one of:
Some Photoshop javascript
A Python function
An OSX API that I can call from python

ANSWER:
Unfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.
Cocoa doesn't have any native support, at least as of 10.5, for getting the location of a font.
' metadata={'question_id': 469, 'answer_id': 3040, 'question_score': 21, 

In [13]:
print(documents[0].metadata)

{'question_id': 469, 'answer_id': 3040, 'question_score': 21, 'answer_score': 12, 'source': 'stackoverflow'}


In [10]:
documents

[Document(metadata={'question_id': 469, 'answer_id': 3040, 'question_score': 21, 'answer_score': 12, 'source': 'stackoverflow'}, page_content="\nTITLE:\nHow can I find the full path to a font from its display name on a Mac?\n\nQUESTION:\nI am using the Photoshop's javascript API to find the fonts in a given PSD.\nGiven a font name returned by the API, I want to find the actual physical font file that that font name corresponds to on the disc.\nThis is all happening in a python program running on OSX so I guess I'm looking for one of:\nSome Photoshop javascript\nA Python function\nAn OSX API that I can call from python\n\nANSWER:\nUnfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.\nCocoa doesn't have any native support, at least as of 10.5, for getting the loc

In [14]:
lengths = [
    len(doc.page_content)
    for doc in documents[:10000]
]

print("Min:", min(lengths))
print("Max:", max(lengths))
print("Avg:", sum(lengths)/len(lengths))

Min: 141
Max: 25560
Avg: 1524.8275


In [15]:
from app.ingestion.chunks_docs import chunk_documents

chunked_docs = chunk_documents(documents[:1000])

print("Original:", len(documents[:1000]))
print("Chunked :", len(chunked_docs))

Original: 1000
Chunked : 1282


In [16]:
import pickle

with open(
    "app/assets/processed/chunked_docs.pkl",
    
    "wb"
) as f:
    pickle.dump(
        chunked_docs,
        f
    )

print(
    f"Saved {len(chunked_docs)} chunks"
)

Saved 1282 chunks


In [17]:
type(chunked_docs[0])

langchain_core.documents.base.Document

In [18]:
chunked_docs[0]

Document(metadata={'question_id': 469, 'answer_id': 3040, 'question_score': 21, 'answer_score': 12, 'source': 'stackoverflow'}, page_content="\nTITLE:\nHow can I find the full path to a font from its display name on a Mac?\n\nQUESTION:\nI am using the Photoshop's javascript API to find the fonts in a given PSD.\nGiven a font name returned by the API, I want to find the actual physical font file that that font name corresponds to on the disc.\nThis is all happening in a python program running on OSX so I guess I'm looking for one of:\nSome Photoshop javascript\nA Python function\nAn OSX API that I can call from python\n\nANSWER:\nUnfortunately the only API that isn't deprecated is located in the ApplicationServices framework, which doesn't have a bridge support file, and thus isn't available in the bridge. If you're wanting to use ctypes, you can use ATSFontGetFileReference after looking up the ATSFontRef.\nCocoa doesn't have any native support, at least as of 10.5, for getting the loca

In [19]:
from app.embeddings.embeddings_model import get_embedding_model

embedding_model = get_embedding_model() 
embedding_model.encode("What is the list comprehension in Python?")

Loading BGE model into memory...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

array([-1.24588248e-03,  2.67008971e-03,  1.58403348e-02, -6.96188211e-02,
        8.35167915e-02,  4.94599603e-02, -1.95017774e-02, -2.78915316e-02,
       -2.83316635e-02,  2.48730765e-04, -2.45398879e-02,  5.35994433e-02,
       -5.94472289e-02, -2.09603123e-02,  1.59503333e-02,  9.55273136e-02,
        5.40709123e-02, -2.23763566e-03, -2.93456409e-02,  1.85204577e-03,
        2.16488987e-02,  1.40367560e-02, -1.15971016e-02, -1.04357079e-02,
       -1.34312538e-02,  2.33157203e-02,  1.39788510e-02, -1.30240815e-02,
       -2.42395885e-02, -8.30471143e-03,  1.18913641e-02,  1.38949342e-02,
       -8.03327840e-03, -1.89843141e-02,  3.91115695e-02,  8.15670472e-03,
        2.35962472e-03, -2.66484171e-02, -3.99145298e-02, -2.65235244e-03,
       -7.89295975e-03, -3.50250788e-02, -5.02503067e-02,  2.16018017e-02,
       -7.41655603e-02,  2.49898201e-03, -2.72420887e-02,  1.59034114e-02,
       -5.89107946e-02,  4.44511045e-03, -3.71399187e-02,  5.91110848e-02,
       -2.02568416e-02, -

In [20]:
from app.vector_store.index_docs import create_collection

create_collection()

Connecting to Qdrant
Collection Already Exists


In [11]:
from app.vector_store.index_docs import (
    create_collection,
    index_documents
)

from app.ingestion.creat_doc import create_documents

# temporary smoke test

create_collection()

# sample_docs = documents[:1000]

index_documents(documents)

print("Smoke Test Passed")

2026-06-28 11:06:28,849 | INFO | python_tutor_rag | Initializing Qdrant client...
Collection Already Exists
2026-06-28 11:06:28,960 | INFO | python_tutor_rag | Loading BGE model into memory...


2026-06-28 11:06:30,674 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-28 11:06:35,790 | INFO | python_tutor_rag | Model id: 17916279312


KeyboardInterrupt: 

In [22]:
from app.retriever.retriever import retrieve


results = retrieve(
    "What is the difference between list and tuple?"
)

for r in results:
    print(r)
    print("=" * 100)

2026-06-27 21:52:16,805 | INFO | python_tutor_rag | Hybrid retrieval started for query: What is the difference between list and tuple?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-27 21:52:17,183 | INFO | httpx | HTTP Request: POST http://localhost:6333/collections/stackoverflow_python/points/query "HTTP/1.1 200 OK"
2026-06-27 21:52:17,195 | INFO | python_tutor_rag | Retrieved 5 chunks
{'content': 'TITLE:\nWhat is the difference between @staticmethod and @classmethod in Python?\n\nQUESTION:\nWhat is the difference between a function decorated with\n[CODE]\n@staticmethod\n[/CODE]\nand one decorated with\n[CODE]\n@classmethod\n[/CODE]\n?', 'question_id': 136097, 'answer_id': 1669524, 'question_score': 1583, 'answer_score': 1256, 'source': 'stackoverflow', 'chunk_id': 0, 'parent_question_id': 136097}
{'content': "TITLE:\nWhat is the difference between Python's re.search and re.match?\n\nQUESTION:\nWhat is the difference between the\n[CODE]\nsearch()\n[/CODE]\nand\n[CODE]\nmatch()\n[/CODE]\nfunctions in the\nPython\n[CODE]\nre\n[/CODE]\nmodule\n?\nI've read the\ndocumentation\n(\ncurrent documentation\n), but I never seem to remember it.  I keep having to lo

In [26]:
from app.retriever.load_bm25 import bm25_retriever

results = bm25_retriever.retrieve(
    "difference between append and extend",
    top_k=5
)

print(type(results[0]))
print(results[0])

<class 'dict'>
{'content': 'TITLE:\nWhat is the difference between @staticmethod and @classmethod in Python?\n\nQUESTION:\nWhat is the difference between a function decorated with\n[CODE]\n@staticmethod\n[/CODE]\nand one decorated with\n[CODE]\n@classmethod\n[/CODE]\n?', 'question_id': 136097, 'answer_id': 1669524, 'question_score': 1583, 'answer_score': 1256, 'source': 'stackoverflow', 'chunk_id': 0, 'parent_question_id': 136097}


In [25]:
from app.retriever.retriever import retrieve
results = retrieve(
    "difference between append and extend"
)

print(results[0])

2026-06-27 21:53:09,185 | INFO | python_tutor_rag | Hybrid retrieval started for query: difference between append and extend


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-27 21:53:10,041 | INFO | httpx | HTTP Request: POST http://localhost:6333/collections/stackoverflow_python/points/query "HTTP/1.1 200 OK"
2026-06-27 21:53:10,055 | INFO | python_tutor_rag | Retrieved 5 chunks
{'content': 'TITLE:\nWhat is the difference between @staticmethod and @classmethod in Python?\n\nQUESTION:\nWhat is the difference between a function decorated with\n[CODE]\n@staticmethod\n[/CODE]\nand one decorated with\n[CODE]\n@classmethod\n[/CODE]\n?', 'question_id': 136097, 'answer_id': 1669524, 'question_score': 1583, 'answer_score': 1256, 'source': 'stackoverflow', 'chunk_id': 0, 'parent_question_id': 136097}


In [34]:
from ddgs import DDGS

with DDGS() as ddgs:
    results = list(
        ddgs.text(
            "difference between decorator and itrerator python",
            max_results=3
        )
    )

print(results)

2026-06-27 22:22:59,612 | INFO | primp | response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=difference%20between%20decorator%20and%20itrerator%20python 200
2026-06-27 22:23:00,223 | INFO | primp | response: https://grokipedia.com/api/typeahead?query=difference+between+decorator+and+itrerator+python&limit=1 200
2026-06-27 22:23:01,078 | INFO | primp | response: https://www.google.com/search?q=difference+between+decorator+and+itrerator+python&filter=1&start=0&hl=en-US&lr=lang_en&cr=countryUS 200
[{'title': 'What are Iterators, Generators And Decorators in Python?', 'href': 'https://rishikonapure.medium.com/what-are-iterators-generators-and-decorators-in-python-d3f9064184c6', 'body': '13 Apr 2022 · A decorator in Python is any callable Python object that is used to modify a function or a class. It takes in a function, adds some ...'}, {'title': 'Python Iterators and Generators Tutorial - DataCamp', 'href': 'https://www.datacamp.com/tutorial/python-